# EDA: IBM TabFormer credit card transactions

Realistic **synthetic** data from IBM (2,000 people, 6,139 cards, 24.4 million transactions, 1991 to 2020).
Loading only the first 3 million rows (about 252 people, with their complete histories) so it fits in memory.

Why not PaySim: see `01_eda.ipynb`. Almost every sender appears only once, and the balance columns leak the label.

## 1. Load and look

In [2]:
data_path="D:/Coding/fraud-detection-realtime/data/raw/card_transaction.v1.csv"
import pandas as pd

In [3]:
df = pd.read_csv("../data/raw/card_transaction.v1.csv", nrows=3_000_000)

**Notes:** what does one row represent? Which column is the target?

## 2. Data quality

In [4]:
df.shape

(3000000, 15)

In [34]:
df.head(10)

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud
0,0,0,2002,9,1,06:21,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,0
1,0,0,2002,9,1,06:42,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0
2,0,0,2002,9,2,06:22,120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0
3,0,0,2002,9,2,17:45,128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,0
4,0,0,2002,9,3,06:23,104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,0
5,0,0,2002,9,3,13:53,86.19,Swipe Transaction,-7146670748125200898,Monterey Park,CA,91755.0,5970,NaN,0
6,0,0,2002,9,4,05:51,93.84,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0
7,0,0,2002,9,4,06:09,123.50,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0
8,0,0,2002,9,5,06:14,61.72,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0
9,0,0,2002,9,5,09:35,57.10,Swipe Transaction,4055257078481058705,La Verne,CA,91750.0,7538,NaN,0


In [9]:
df.columns=df.columns.str.lower().str.replace(" ","_")

In [35]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 15 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user            int64  
 1   card            int64  
 2   year            int64  
 3   month           int64  
 4   day             int64  
 5   time            str    
 6   amount          float64
 7   use_chip        str    
 8   merchant_name   int64  
 9   merchant_city   str    
 10  merchant_state  str    
 11  zip             float64
 12  mcc             int64  
 13  errors          str    
 14  is_fraud        int64  
dtypes: float64(2), int64(8), str(5)
memory usage: 437.1 MB


In [14]:
df.amount=df["amount"].str.replace("$","").astype(float)

In [28]:
df.columns=df.columns.str.replace("?","")

In [31]:
df[df["amount"]<0].is_fraud.value_counts()

is_fraud
No     143318
Yes       122
Name: count, dtype: int64

In [32]:
df.is_fraud.value_counts(normalize=True)

is_fraud
No     0.99886
Yes    0.00114
Name: proportion, dtype: float64

In [33]:
df["is_fraud"]=(df["is_fraud"]=="Yes").astype(int)

In [41]:
df.head(3)

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
0,0,0,2002,9,1,06:21,134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,0,2002-9-1 06:21
1,0,0,2002,9,1,06:42,38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0,2002-9-1 06:42
2,0,0,2002,9,2,06:22,120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,0,2002-9-2 06:22


In [40]:
df["timestamp"]=df["year"].astype(str)+"-"+df["month"].astype(str)+"-"+df["day"].astype(str)+" "+df["time"]

In [42]:
df["timestamp"]=pd.to_datetime(df["timestamp"],format= "%Y-%m-%d %H:%M")

In [43]:
df["timestamp"].dtype

dtype('<M8[us]')

In [44]:
df["timestamp"].min(), df["timestamp"].max()

(Timestamp('1995-06-18 19:25:00'), Timestamp('2020-02-28 23:49:00'))

In [45]:
df.isna().sum()

user                    0
card                    0
year                    0
month                   0
day                     0
time                    0
amount                  0
use_chip                0
merchant_name           0
merchant_city           0
merchant_state     378407
zip                399736
mcc                     0
errors            2953851
is_fraud                0
timestamp               0
dtype: int64

In [48]:
df[df.duplicated()]

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
193878,13,0,2015,4,28,06:36,100.0,Chip Transaction,-4282466774399734331,Boyne City,MI,49712.0,4829,"Insufficient Balance,",0,2015-04-28 06:36:00
195221,13,1,2006,7,19,07:34,100.0,Swipe Transaction,-4282466774399734331,Boyne City,MI,49712.0,4829,"Insufficient Balance,",0,2006-07-19 07:34:00
195330,13,1,2006,10,15,07:11,100.0,Swipe Transaction,-4282466774399734331,Boyne City,MI,49712.0,4829,"Insufficient Balance,",0,2006-10-15 07:11:00
244172,17,2,2011,12,12,16:26,120.0,Swipe Transaction,-4282466774399734331,Prescott Valley,AZ,86314.0,4829,"Insufficient Balance,",0,2011-12-12 16:26:00
1360769,109,2,1999,12,9,04:16,120.0,Swipe Transaction,-4282466774399734331,Hammond,IN,46323.0,4829,"Insufficient Balance,",0,1999-12-09 04:16:00
1373587,109,2,2019,3,8,04:22,120.0,Chip Transaction,-4282466774399734331,Hammond,IN,46323.0,4829,"Insufficient Balance,",0,2019-03-08 04:22:00
2469536,205,2,2012,2,8,17:33,82.0,Swipe Transaction,2027553650310142703,Huntington Station,NY,11746.0,5541,NaN,0,2012-02-08 17:33:00
2773484,235,1,2008,5,22,10:23,83.0,Swipe Transaction,1799189980464955940,Garland,TX,75042.0,5499,NaN,0,2008-05-22 10:23:00


In [49]:
df["use_chip"].value_counts()

use_chip
Swipe Transaction     1886840
Chip Transaction       735675
Online Transaction     377485
Name: count, dtype: int64

In [50]:
no_zip=df[df["zip"].isna() & df["merchant_state"].notna()]

In [51]:
no_zip.head()

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
407,0,0,2003,2,24,13:02,7.52,Swipe Transaction,-727612092139916043,Zurich,Switzerland,NaN,5411,NaN,0,2003-02-24 13:02:00
408,0,0,2003,2,25,06:46,36.97,Swipe Transaction,-4816289482172287511,Zurich,Switzerland,NaN,5411,NaN,0,2003-02-25 06:46:00
906,0,0,2004,7,22,06:32,38.26,Swipe Transaction,3510944833163794547,Tallinn,Estonia,NaN,5912,NaN,0,2004-07-22 06:32:00
907,0,0,2004,7,23,20:05,75.38,Swipe Transaction,8793545055147237096,Tallinn,Estonia,NaN,5812,NaN,0,2004-07-23 20:05:00
908,0,0,2004,7,25,13:08,16.20,Swipe Transaction,2027553650310142703,Tallinn,Estonia,NaN,5541,NaN,0,2004-07-25 13:08:00


In [54]:
no_zip["merchant_state"].value_counts().head(10)

merchant_state
Mexico                6006
Canada                2715
Italy                 1367
United Kingdom        1272
France                 833
China                  765
Germany                753
India                  438
Japan                  421
Dominican Republic     396
Name: count, dtype: int64

## 3. Target

In [55]:
df["is_fraud"].value_counts()

is_fraud
0    2996579
1       3421
Name: count, dtype: int64

In [57]:
df[df["is_fraud"]==1]["user"].nunique()

163

In [58]:
df[df["is_fraud"] == 1].index[0]

np.int64(4099)

In [59]:
df.loc[4099:4099+6]

,user,card,year,month,day,time,amount,use_chip,merchant_name,merchant_city,merchant_state,zip,mcc,errors,is_fraud,timestamp
4099,0,0,2015,11,15,12:55,287.13,Online Transaction,-8194607650924472520,ONLINE,NaN,NaN,3001,NaN,1,2015-11-15 12:55:00
4100,0,0,2015,11,15,13:19,2.41,Online Transaction,-7759074308363763111,ONLINE,NaN,NaN,5651,NaN,1,2015-11-15 13:19:00
4101,0,0,2015,11,16,09:41,50.81,Online Transaction,-551332107213382088,ONLINE,NaN,NaN,4411,NaN,1,2015-11-16 09:41:00
4102,0,0,2015,11,16,09:46,248.36,Online Transaction,4872340518840476610,ONLINE,NaN,NaN,5732,NaN,1,2015-11-16 09:46:00
4103,0,0,2015,11,16,11:20,473.00,Online Transaction,-8566951830324093739,ONLINE,NaN,NaN,3640,NaN,1,2015-11-16 11:20:00
4104,0,0,2015,11,16,11:34,144.49,Online Transaction,-3650057525483423997,ONLINE,NaN,NaN,3007,NaN,1,2015-11-16 11:34:00
4105,0,0,2015,11,16,13:44,44.57,Online Transaction,1913477460590765860,ONLINE,NaN,NaN,5300,NaN,1,2015-11-16 13:44:00


## 4. One column at a time

## 5. Feature against target

## 6. Weird things and leakage checks

## 7. Findings